# **Explaining an agent: the contribution of tools and steps**

A practice for the module [«Explaining agent behaviour»](https://open-xai-platform.web.app).

A real agent run costs money and needs keys, and it is non-deterministic on top of that — two
runs give two different traces. For a practice that is bad: it is unclear what we are measuring,
the method or the variance of the model. So the agent here is a **simulator with a known
structure**: we know in advance which tool affects what, and we check the method against that
knowledge.

The device is not made up: this is how any explanation is checked — first on synthetic data
where the right answer is known, and only then on real data where it is not.

What we will do:

1. compute Shapley values over the tools **exactly**, enumerating all coalitions — this is
   AgentSHAP without Monte Carlo;
2. replace the enumeration with sampling and see how many runs are needed and how much the
   estimate varies;
3. see what the method does not see: a useless tool, an interaction of two tools, and where it
   parts ways with the naive «switch them off one by one»;
4. compute the counterfactual importance of reasoning steps — Thought Anchors in miniature;
5. try to localize a failure in a trace and understand why the numbers for that task are so low.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations, permutations
from math import factorial

rng = np.random.default_rng(0)

TOOLS = ["search", "db", "calc", "translate", "calendar"]


def agent_quality(subset):
    """The quality of the agent answer given the available set of tools. The structure is
    known: search is required — without it the agent does not answer at all;
    db and calc are useful separately and give another 0.25 together — an interaction;
    calendar gives a little, translate gives nothing."""
    S = set(subset)
    if "search" not in S:
        return 0.0
    q = 0.40
    q += 0.20 if "db" in S else 0.0
    q += 0.10 if "calc" in S else 0.0
    q += 0.25 if {"db", "calc"} <= S else 0.0
    q += 0.05 if "calendar" in S else 0.0
    return q


print("all tools:   ", agent_quality(TOOLS))
print("without search:", agent_quality([t for t in TOOLS if t != "search"]))
print("search only: ", agent_quality(["search"]))

## 1. The exact Shapley value: enumerating the coalitions

The formula is the one from the SHAP block, only the «feature» is the availability of a tool.
The contribution of a tool $t$ is the average over all coalitions $S$ it is added to:

$$\varphi_t = \sum_{S \subseteq N \setminus \{t\}} \frac{|S|!\,(n - |S| - 1)!}{n!}\,
\bigl[v(S \cup \{t\}) - v(S)\bigr]$$

Here $v(S)$ is the quality of the answer of an agent that only has the tools from $S$. For a
real agent every such $v(S)$ is a **full run**, and that is exactly why enumeration is so
expensive.

In [ ]:
def shapley_exact(tools, value):
    """Exact Shapley values: enumeration over all subsets."""
    n = len(tools)
    phi = {}
    for t in tools:
        rest = [x for x in tools if x != t]
        total = 0.0
        for k in range(len(rest) + 1):
            for S in combinations(rest, k):
                weight = factorial(k) * factorial(n - k - 1) / factorial(n)
                total += weight * (value(set(S) | {t}) - value(S))
        phi[t] = total
    return phi


phi = shapley_exact(TOOLS, agent_quality)
for tool, value in sorted(phi.items(), key=lambda kv: -kv[1]):
    print(f"{tool:10} {value:6.4f}")

print("\ncoalitions enumerated:", 2 ** len(TOOLS))
print("sum of contributions =", round(sum(phi.values()), 6), "|", agent_quality(TOOLS))

Note the last line: **the sum of the contributions equals the quality of the full set**.
That is the efficiency axiom, and it is also the fastest check of the implementation. If the sum
does not add up to $v(N) - v(\emptyset)$, the implementation is broken.

Five tools is 32 coalitions, and enumeration is still possible. Ten tools is 1024, twenty is over
a million agent runs. Hence Monte Carlo.

## 2. Monte Carlo: how many runs are needed

The permutation estimator: take a random order of the tools, add them one by one and record the
gain in quality. The average over many permutations converges to the Shapley value.

Below is the error of the estimate as a function of the number of permutations, averaged over 20
independent runs. Look at the slope: the error falls as $1/\sqrt{n}$, that is, **four times the
cost for twice the accuracy**.

In [ ]:
def shapley_mc(tools, value, n_samples, rng):
    """The Shapley estimate over random permutations of the tools."""
    phi = {t: 0.0 for t in tools}
    for _ in range(n_samples):
        order = list(rng.permutation(tools))
        before = set()
        v_before = value(before)
        for t in order:
            before.add(t)
            v_after = value(before)
            phi[t] += v_after - v_before
            v_before = v_after
    return {t: v / n_samples for t, v in phi.items()}


sizes = [10, 30, 100, 300, 1000, 3000]
errors = []
for n in sizes:
    runs = [shapley_mc(TOOLS, agent_quality, n, np.random.default_rng(seed)) for seed in range(20)]
    err = np.mean([max(abs(r[t] - phi[t]) for t in TOOLS) for r in runs])
    errors.append(err)
    print(f"{n:>5} {'permutations → error'} {err:.4f}")

plt.figure(figsize=(6, 3.4))
plt.plot(sizes, errors, marker="o")
plt.xscale("log")
plt.yscale("log")
plt.xlabel("permutations")
plt.ylabel("maximum error")
plt.title("Convergence of the Monte Carlo estimate")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Now the thing the authors of AgentSHAP check separately and that is always worth checking:
**the stability of the estimate across runs**. Below are eight independent estimates on a hundred
permutations.

In [ ]:
runs = [shapley_mc(TOOLS, agent_quality, 100, np.random.default_rng(seed)) for seed in range(8)]
for tool in TOOLS:
    values = [r[tool] for r in runs]
    print(f"{tool:10} {np.mean(values):6.4f} ± {np.std(values):.4f}   "
          f"[{min(values):.3f}, {max(values):.3f}]")

Look at the ranges of `db` and `calc`: they overlap. On a hundred permutations a single
run may well rank `calc` above `db` — while the exact values say the opposite. That is the practical
conclusion: until the spread is measured, **the difference between two neighbouring tools may be
sampling noise** rather than a property of the agent.

## 3. What the method does not see

Let us compare the Shapley values with the naive «switch them off one by one» — leave-one-out,
the difference between the quality of the full set and the set without the tool.

In [ ]:
loo = {t: agent_quality(TOOLS) - agent_quality([x for x in TOOLS if x != t]) for t in TOOLS}

print(f"{'tool':10} {'Shapley':>10} {'leave-one-out':>10}")
for t in TOOLS:
    print(f"{t:10} {phi[t]:10.4f} {loo[t]:10.4f}")

print("\nsum of leave-one-out:", round(sum(loo.values()), 4))
print("sum of Shapley:      ", round(sum(phi.values()), 4))

Three things are visible at once.

**`translate` got exactly zero.** That is the dummy axiom: a tool that changes nothing in any
coalition gets no contribution. The good news is that the method does not assign importance to
something that is merely present in the set.

**The sum of leave-one-out does not equal the quality of the full set.** For Shapley it does, for
leave-one-out it does not, and the difference is exactly the size of the interaction. The naive
approach loses what `db` and `calc` give only together: switching them off one by one, you do not
see the joint effect.

**`search` got more than half.** It is required: without it the quality is zero in any coalition.
Shapley reflects that honestly, but it also shows a limitation of the method — a required element
takes the contribution that in some sense belongs to the whole system.

## 4. Reasoning steps: Thought Anchors in miniature

Now a different player — not a tool but a **reasoning step**. The method from the paper: replace a
step with one that differs in meaning and see how the distribution of final answers changed. Here
the trace is fixed, and the «model» is a simulator where it is known which step affects success by
how much.

In [ ]:
TRACE = ["Plan: first pin down the period, then compute", "The user asks about revenue", "Call db_lookup: revenue by quarter", "Four numbers received", "Let me check I am not confusing a quarter with a half-year", "Answer: a 12 per cent growth"]

# The effect of a step on success — what the method has to recover
EFFECT = [0.45, 0.05, 0.30, 0.02, 0.10, 0.03]


def run_from(step_index, replaced, rng, n=400):
    """The share of successful completions if step step_index is replaced by another one."""
    p = 0.9
    for i, effect in enumerate(EFFECT):
        if i == step_index and replaced:
            p -= effect
    p = float(np.clip(p, 0.0, 1.0))
    return rng.random(n) < p


base = run_from(-1, False, np.random.default_rng(0)).mean()
importance = []
for i in range(len(TRACE)):
    changed = run_from(i, True, np.random.default_rng(100 + i)).mean()
    importance.append(base - changed)
    print(f"{i + 1}. {TRACE[i][:46]:48} {base - changed:6.3f}")

plt.figure(figsize=(6.4, 3.2))
plt.barh(range(len(TRACE), 0, -1), importance)
plt.yticks(range(len(TRACE), 0, -1), [f"{i + 1}" for i in range(len(TRACE))])
plt.xlabel("drop in success rate when the step is replaced")
plt.title("Counterfactual importance of the steps")
plt.tight_layout()
plt.show()

The first and the fifth steps came out on top — planning and managing one’s own
uncertainty, rather than the step where the computation is done. That is the finding of the paper:
**the anchors turn out to be the steps where the path is chosen**, not the ones where the answer is
computed.

Note the honesty of the setup: we replace a step with **one that differs in meaning**. If we
substituted a rephrasing of the same thing, the method would measure robustness to paraphrase
rather than the influence of the step.

## 5. Localizing a failure: why the numbers are so low

The last task is not explanation but diagnostics: in the trace of a multi-agent system, find
**which** agent and **at which step** broke the run. Let us check two naive heuristics on synthetic
traces where the culprit is known.

In [ ]:
def make_trace(rng, n_agents=4, n_steps=12):
    """A trace of 12 steps and 4 agents; the fault is injected at a known step by a known agent."""
    culprit = int(rng.integers(n_agents))
    step = int(rng.integers(2, n_steps - 2))
    owners = rng.integers(0, n_agents, size=n_steps)
    owners[step] = culprit
    return owners, culprit, step


def guess_last(owners):
    """Naive heuristic: the one who spoke last is to blame."""
    return int(owners[-1]), len(owners) - 1


def guess_middle(owners):
    """Naive heuristic: the failure is in the middle of the trace."""
    return int(owners[len(owners) // 2]), len(owners) // 2


rng2 = np.random.default_rng(7)
hits = {"last": [0, 0], "middle": [0, 0]}
N = 2000
for _ in range(N):
    owners, culprit, step = make_trace(rng2)
    for name, guess in (("last", guess_last), ("middle", guess_middle)):
        who, when = guess(owners)
        hits[name][0] += who == culprit
        hits[name][1] += when == step

for name, (who, when) in hits.items():
    print(f"{name:8} {'agent:'} {who / N:6.1%}   {'step:'} {when / N:6.1%}")

print(f"\n{'random guessing of the agent:'} {1 / 4:.1%}")

Both heuristics work at the level of random guessing for the agent, and for the step they
practically never hit. That is the reason behind the numbers from the lesson: the best method in
the Who&When study gets **53.5 %** for the agent and **14.2 %** for the step, and strong reasoning
models on the same task get below 10 %. The «who and when» task is an order of magnitude harder
than «what happened», and no tracer closes it today.

## Exercises

Submit the answers on Stepik — in the homework step of this module.

1. How many coalitions would have to be enumerated for the exact Shapley value if the agent had
   **eight** tools?
2. What does $\varphi$ of the tool `search` equal? Round to two decimal places.
3. What does the sum of all the Shapley values equal, and why exactly that number?
4. Add to `agent_quality` a sixth tool that duplicates `db` (gives exactly the same when `db` is
   already there, and the same in its place when `db` is not). Recompute the values. What happened
   to the contribution of `db`, and why is that the correct behaviour of the method?
5. In section 4, replace «replacing a step with one that differs in meaning» with «replacing it
   with a rephrasing of the same step» — that is, make the effect of the replacement zero. What
   will the method show, and why does that not mean the steps are unimportant?